# Silhouette Score Demonstration

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/klar74/WS2026_lecture/blob/main/Vorlesung_11/Silhouette_Score_Demonstration.ipynb)

Dieses Notebook zeigt anhand verschiedener Clustering-Szenarien, wie der Silhouette Score funktioniert und was er über die Qualität der Cluster aussagt.

**Erinnerung an Vorlesung 11:**
- **a:** Durchschnittlicher Abstand zu allen *anderen* Punkten im eigenen Cluster (soll klein sein)
- **b:** Kleinster durchschnittlicher Abstand zu den Punkten eines anderen Clusters (soll groß sein)
- **Formel:** $s = \frac{b - a}{\max(a, b)}$, wobei $s \in [-1, +1]$
- **s ≈ +1:** Punkt liegt klar in seinem Cluster (gut!)
- **s ≈ 0:** Punkt liegt zwischen zwei Clustern (grenzwertig)
- **s ≈ -1:** Punkt liegt im Mittel näher an einem anderen Cluster (Hinweis auf eine ungünstige Zuordnung)

Der **Silhouette Score** ist der Mittelwert dieser Punktwerte. Hier verwenden wir euklidische Abstände.

Für den Silhouette Score benötigen wir mindestens zwei Cluster. Außerdem darf nicht jeder Datenpunkt einen eigenen Cluster bilden. Einzelne Punkte, die allein in ihrem Cluster liegen, erhalten in scikit-learn den Silhouette-Wert 0.

**Wichtig:** Die folgenden Bewertungsgrenzen sind grobe Orientierung, keine allgemeingültigen Qualitätsnachweise. Der Score hängt von Distanzmaß, Skalierung und Clusterform ab. Ein hoher Wert beweist keine fachlich sinnvollen Gruppen; ein niedriger Wert schließt andere, etwa nicht-konvexe Strukturen nicht aus. Alle Zellen bitte von oben nach unten ausführen.

In [ ]:
# Import der benötigten Bibliotheken
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.preprocessing import StandardScaler
import seaborn as sns

# Stil für bessere Plots
plt.style.use('default')
sns.set_palette("husl")

def silhouette_quality(score):
    if score >= 0.7:
        return 'Sehr starke Trennung'
    if score >= 0.5:
        return 'Deutliche Trennung'
    if score >= 0.25:
        return 'Schwache bis mäßige Trennung'
    return 'Geringe oder ungünstige Trennung'

print("✓ Bibliotheken erfolgreich importiert")

## Szenario 1: Perfekt getrennte Cluster (Erwartung: hoher Silhouette Score)

Wir erstellen drei klar getrennte, kompakte Cluster - die ideale Situation für Clustering.

In [ ]:
# Perfekt getrennte Cluster erstellen
X_perfect, y_true_perfect = make_blobs(
    n_samples=300, 
    centers=3, 
    cluster_std=0.8,  # Kleine Standardabweichung = kompakte Cluster
    center_box=(-10.0, 10.0),  # Große Bounding Box = weit auseinander
    random_state=42
)

# K-Means Clustering
kmeans_perfect = KMeans(n_clusters=3, random_state=42, n_init=10)
y_pred_perfect = kmeans_perfect.fit_predict(X_perfect)

# Silhouette Score berechnen
sil_score_perfect = silhouette_score(X_perfect, y_pred_perfect)
sil_samples_perfect = silhouette_samples(X_perfect, y_pred_perfect)

print(f"Perfekte Cluster - Silhouette Score: {sil_score_perfect:.3f}")
print(f"Orientierung: {silhouette_quality(sil_score_perfect)}")

In [ ]:
# Visualisierung der perfekten Cluster
plt.figure(figsize=(10, 7))

# Scatter Plot mit Clustern
colors = ['red', 'blue', 'green']
for i in range(3):
    mask = y_pred_perfect == i
    plt.scatter(X_perfect[mask, 0], X_perfect[mask, 1], 
               c=colors[i], alpha=0.7, s=50, 
               label=f'Cluster {i+1}')

# Cluster-Zentren markieren
centers = kmeans_perfect.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1], 
           c='black', marker='x', s=200, linewidths=3, label='Zentren')

plt.title(f'Perfekte Cluster\nSilhouette Score: {sil_score_perfect:.3f} ({silhouette_quality(sil_score_perfect)})', fontsize=14)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Orientierung: {silhouette_quality(sil_score_perfect)}")
print(f"   Die Cluster sind klar getrennt und kompakt")

## Szenario 2: Überlappende Cluster (Erwartung: mittlerer Silhouette Score)

Hier erstellen wir Cluster, die sich teilweise überlappen - eine häufige Situation in realen Daten.

In [ ]:
# Überlappende Cluster erstellen
X_overlap, y_true_overlap = make_blobs(
    n_samples=300, 
    centers=3, 
    cluster_std=2.5,  # Größere Standardabweichung = überlappende Cluster
    center_box=(-5.0, 5.0),  # Kleinere Bounding Box = näher zusammen
    random_state=42
)

# K-Means Clustering
kmeans_overlap = KMeans(n_clusters=3, random_state=42, n_init=10)
y_pred_overlap = kmeans_overlap.fit_predict(X_overlap)

# Silhouette Score berechnen
sil_score_overlap = silhouette_score(X_overlap, y_pred_overlap)
sil_samples_overlap = silhouette_samples(X_overlap, y_pred_overlap)

print(f"Überlappende Cluster - Silhouette Score: {sil_score_overlap:.3f}")
print(f"Orientierung: {silhouette_quality(sil_score_overlap)}")

In [ ]:
# Visualisierung der überlappenden Cluster
plt.figure(figsize=(10, 7))

# Scatter Plot
for i in range(3):
    mask = y_pred_overlap == i
    plt.scatter(X_overlap[mask, 0], X_overlap[mask, 1], 
               c=colors[i], alpha=0.6, s=50, 
               label=f'Cluster {i+1}')

centers = kmeans_overlap.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1], 
           c='black', marker='x', s=200, linewidths=3, label='Zentren')

plt.title(f'Überlappende Cluster\nSilhouette Score: {sil_score_overlap:.3f} ({silhouette_quality(sil_score_overlap)})', fontsize=14)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Orientierung: {silhouette_quality(sil_score_overlap)}")
print(f"   Die Cluster überlappen - weniger klare Trennung")

## Szenario 3: Falsche Anzahl Cluster (Erwartung: niedriger Silhouette Score)

Was passiert, wenn wir zu viele Cluster für die Datenstruktur wählen? Wir nehmen die perfekten Daten von Szenario 1, aber clustern in 6 statt 3 Gruppen.

In [ ]:
# Zu viele Cluster für die gleichen perfekten Daten
kmeans_wrong = KMeans(n_clusters=6, random_state=42, n_init=10)  # 6 statt 3
y_pred_wrong = kmeans_wrong.fit_predict(X_perfect)

# Silhouette Score berechnen
sil_score_wrong = silhouette_score(X_perfect, y_pred_wrong)
sil_samples_wrong = silhouette_samples(X_perfect, y_pred_wrong)

print(f"Falsche Cluster-Anzahl - Silhouette Score: {sil_score_wrong:.3f}")
print(f"Interpretation: Score deutlich niedriger als bei korrekter Anzahl ({sil_score_perfect:.3f})")
print(f"Grund: Natürliche Cluster werden künstlich aufgespalten")

In [ ]:
# Visualisierung der falschen Cluster-Anzahl
plt.figure(figsize=(10, 7))

# Scatter Plot mit 6 Clustern
colors_6 = ['red', 'blue', 'green', 'orange', 'purple', 'yellow']
for i in range(6):
    mask = y_pred_wrong == i
    plt.scatter(X_perfect[mask, 0], X_perfect[mask, 1], 
               c=colors_6[i], alpha=0.7, s=50, 
               label=f'Cluster {i+1}')

centers = kmeans_wrong.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1], 
           c='black', marker='x', s=200, linewidths=3, label='Zentren')

plt.title(f'Falsche Cluster-Anzahl (6 statt 3)\nSilhouette Score: {sil_score_wrong:.3f} (Schlechter!)', fontsize=14)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"❌ Score deutlich niedriger als bei korrekter Anzahl ({sil_score_perfect:.3f})")
print(f"   Grund: Natürliche Cluster werden künstlich aufgespalten")

## Szenario 4: k-Werte mit dem Silhouette Score vergleichen

Wir testen k = 2 bis 7 mit K-Means und wählen den besten getesteten k-Wert nach dem Silhouette Score. Das ist keine Garantie für eine allgemein optimale Cluster-Anzahl: Das Ergebnis hängt von Daten, Skalierung, Verfahren und Suchbereich ab. Hier kennen wir die drei Gruppen aus der Datenerzeugung und können das Ergebnis damit vergleichen.

In [ ]:
# Verschiedene k-Werte testen
k_range = range(2, 8)  # Von 2 bis 7 Cluster
silhouette_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_perfect)
    score = silhouette_score(X_perfect, cluster_labels)
    silhouette_scores.append(score)
    print(f"k={k}: Silhouette Score = {score:.3f}")

# Besten getesteten k-Wert nach Silhouette auswählen
best_k = k_range[np.argmax(silhouette_scores)]
max_score = max(silhouette_scores)

print(f"\n🎯 Bester getesteter k-Wert: {best_k} (Score: {max_score:.3f})")

In [ ]:
# Silhouette Score vs. Anzahl Cluster plotten
plt.figure(figsize=(10, 6))
plt.plot(k_range, silhouette_scores, 'bo-', linewidth=2, markersize=8)
plt.axvline(x=best_k, color='red', linestyle='--', alpha=0.7, 
           label=f'Bester getesteter Wert: k = {best_k}')
plt.axhline(y=max_score, color='red', linestyle='--', alpha=0.7)

# Annotationen
plt.annotate(f'Maximum: k={best_k}\nScore={max_score:.3f}', 
            xy=(best_k, max_score), 
            xytext=(best_k+0.5, max_score-0.05),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=12, ha='left')

plt.xlabel('Anzahl Cluster (k)', fontsize=12)
plt.ylabel('Silhouette Score', fontsize=12)
plt.title('Vergleich getesteter k-Werte nach Silhouette', fontsize=14)
plt.grid(True, alpha=0.3)
plt.xticks(k_range)

# Grobe Orientierungsbereiche, keine allgemeinen Qualitätsnachweise
plt.axhspan(0.7, 1.0, alpha=0.1, color='green', label='Sehr starke Trennung (ab 0.7)')
plt.axhspan(0.5, 0.7, alpha=0.1, color='yellow', label='Deutliche Trennung (0.5 bis <0.7)')
plt.axhspan(0.25, 0.5, alpha=0.1, color='orange', label='Schwache bis mäßige Trennung (0.25 bis <0.5)')
plt.axhspan(-1.0, 0.25, alpha=0.1, color='red', label='Geringe oder ungünstige Trennung (<0.25)')
plt.legend(title='Grobe Orientierung', fontsize=9)

plt.tight_layout()
plt.show()

## Szenario 5: Smart Operations Beispiel - Maschinennutzungsprofile

Praktisches Beispiel aus der Vorlesung: Verschiedene Maschinennutzungsprofile basierend auf Betriebszeit (h/Tag) und Auslastung (%).

Für die einfache Simulation ziehen wir normalverteilte Werte und begrenzen sie anschließend ausdrücklich auf 0 bis 24 Stunden bzw. 0 bis 100 Prozent. Das ist ein vereinfachtes, an den Grenzen abgeschnittenes Modell, keine unveränderte Normalverteilung. Bei realen Daten würden wir unplausible Werte zunächst untersuchen, nicht einfach begrenzen.

In [ ]:
# Simulierte Maschinendaten erstellen
np.random.seed(42)

# Dauerbetrieb: Hohe Betriebszeit, hohe Auslastung
dauerbetrieb_x = np.random.normal(20, 2, 80)  # ~20h Betriebszeit
dauerbetrieb_y = np.random.normal(85, 5, 80)  # ~85% Auslastung

# Teilzeit: Mittlere Betriebszeit, variable Auslastung
teilzeit_x = np.random.normal(10, 3, 80)  # ~10h Betriebszeit
teilzeit_y = np.random.normal(60, 10, 80)  # ~60% Auslastung

# Niedrige Nutzung: Niedrige Betriebszeit und Auslastung
niedrige_nutzung_x = np.random.normal(4, 2, 40)  # ~4h Betriebszeit
niedrige_nutzung_y = np.random.normal(30, 15, 40)  # ~30% Auslastung

# Daten kombinieren
X_machines = np.vstack([
    np.column_stack([dauerbetrieb_x, dauerbetrieb_y]),
    np.column_stack([teilzeit_x, teilzeit_y]),
    np.column_stack([niedrige_nutzung_x, niedrige_nutzung_y])
])
X_machines[:, 0] = np.clip(X_machines[:, 0], 0, 24)
X_machines[:, 1] = np.clip(X_machines[:, 1], 0, 100)

# Echte Labels für Vergleich (nur zur Evaluation, nicht für Clustering)
y_true_machines = np.array([0]*80 + [1]*80 + [2]*40)

print(f"Maschinendaten erstellt: {X_machines.shape[0]} Maschinen")
print(f"Features: [Betriebszeit (h/Tag), Auslastung (%)]")

### Aufgabe: Nutzungsprofile ohne vorgegebene Labels untersuchen

**Stell dir vor:** Du bist Produktionsleiter und hast Daten von 200 Maschinen. Du weißt nur:
- Wie viele Stunden pro Tag jede Maschine läuft
- Wie stark sie dabei ausgelastet ist (in Prozent)

**Die Frage:** Gibt es Gruppen von Maschinen mit ähnlichen Nutzungsmustern, ohne dass die simulierten Gruppenlabels in das Clustering eingehen?

**Clustering-Ziel:** Ähnliche Nutzungsprofile als Ausgangspunkt für die Ressourcenplanung finden. Niedrige Nutzung kann viele Ursachen haben, etwa geringe Nachfrage, geplanten Stillstand oder Störungen; sie ist keine Diagnose eines Wartungsfalls.

**Vorverarbeitung:** Stunden und Prozent haben unterschiedliche Einheiten und Streuungen. Wir standardisieren beide Merkmale mit `StandardScaler`, damit keines allein wegen seiner größeren numerischen Streuung dominiert. Dies gewichtet beide Merkmale nach ihrer Standardabweichung; fachlich können andere Gewichtungen sinnvoll sein. K-Means und Silhouette verwenden dieselben standardisierten Daten. Für die Diagramme transformieren wir die Zentren in Originaleinheiten zurück.

Wir geben hier **k = 3** zur Demonstration vor, passend zu den drei simulierten Profilen. K-Means entdeckt nicht selbstständig die Anzahl der Gruppen. Die Namen werden erst nach dem Clustering anhand der mittleren Betriebszeit vergeben; 7 und 15 Stunden sind illustrative, von uns gewählte Grenzen.

In [ ]:
# Rohdaten visualisieren - ohne Clustering, nur die beiden Features
plt.figure(figsize=(10, 7))
plt.scatter(X_machines[:, 0], X_machines[:, 1], 
           alpha=0.6, s=50, c='gray', edgecolors='black', linewidth=0.5)

plt.xlabel('Betriebszeit (Stunden/Tag)', fontsize=12)
plt.ylabel('Auslastung (%)', fontsize=12)
plt.title('Rohdaten: 200 Maschinen ohne Gruppierung\nKann ein Algorithmus hier Muster erkennen?', fontsize=14)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Welche Nutzungsmuster vermutest du in den Rohdaten?")
print("Wir untersuchen nun eine Aufteilung in drei Gruppen mit K-Means.")

In [ ]:
scaler_machines = StandardScaler()
X_machines_scaled = scaler_machines.fit_transform(X_machines)

kmeans_machines = KMeans(n_clusters=3, random_state=42, n_init=10)
y_pred_machines = kmeans_machines.fit_predict(X_machines_scaled)

# Silhouette Score berechnen
sil_score_machines = silhouette_score(X_machines_scaled, y_pred_machines)
sil_samples_machines = silhouette_samples(X_machines_scaled, y_pred_machines)

print(f"Maschinennutzungsprofile - Silhouette Score: {sil_score_machines:.3f}")
print(f"Orientierung: {silhouette_quality(sil_score_machines)} (standardisierte Merkmale)")

# Cluster-Labels zuordnen (für bessere Interpretation)
centers_machines = scaler_machines.inverse_transform(kmeans_machines.cluster_centers_)
labels_machines = []
for center in centers_machines:
    if center[0] > 15:  # Hohe Betriebszeit
        labels_machines.append('Dauerbetrieb')
    elif center[0] > 7:  # Mittlere Betriebszeit
        labels_machines.append('Teilzeit')
    else:  # Niedrige Betriebszeit
        labels_machines.append('Niedrige Nutzung')

for i, name in enumerate(labels_machines):
    print(f"  Cluster {i}: {name}")

In [ ]:
# Visualisierung der Maschinennutzungsprofile
fig, ax1 = plt.subplots(1, 1, figsize=(12, 8))

# Scatter Plot der Maschinenprofile
colors_machines = ['darkred', 'darkblue', 'darkgreen']

for i in range(3):
    mask = y_pred_machines == i
    ax1.scatter(X_machines[mask, 0], X_machines[mask, 1], 
               c=colors_machines[i], alpha=0.7, s=60, 
               label=f'Cluster {i}: {labels_machines[i]}')

# Cluster-Zentren
ax1.scatter(centers_machines[:, 0], centers_machines[:, 1], 
           c='black', marker='x', s=300, linewidths=4, label='Zentren')

ax1.set_title(f'Maschinennutzungsprofile (k=3, standardisiert geclustert)\nSilhouette Score: {sil_score_machines:.3f}', fontsize=14)
ax1.set_xlabel('Betriebszeit (Stunden/Tag)', fontsize=12)
ax1.set_ylabel('Auslastung (%)', fontsize=12)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Gesamtwert und Mittelwerte je Cluster
print("🔍 CLUSTER-QUALITÄT ANALYSE")
print("=" * 50)

print(f"\nGesamt-Score: {sil_score_machines:.3f} ({silhouette_quality(sil_score_machines)})")
print("Bewertungen sind grobe Orientierung im standardisierten Merkmalsraum.")

# Pro-Cluster Analyse
print(f"\n📊 CLUSTER-DETAILS:")
for i in range(3):
    cluster_sil_scores = sil_samples_machines[y_pred_machines == i]
    avg_score = np.mean(cluster_sil_scores)
    cluster_size = len(cluster_sil_scores)
    center = centers_machines[i]
    
    cluster_quality = silhouette_quality(avg_score)
    print(f"  {labels_machines[i]:.<20} {avg_score:.3f} ({cluster_size:3d} Maschinen) {cluster_quality}")
    print(f"    Zentrum: {center[0]:.1f}h/Tag, {center[1]:.1f}% Auslastung")

print(f"\n💡 INTERPRETATION:")
print("• Der Gesamtwert kann Unterschiede zwischen Clustern und einzelnen Punkten verdecken.")
print("• Hohe Werte belegen geometrische Trennung, nicht die fachliche Bedeutung der Gruppen.")
print("• Niedrige Werte können auch durch ungeeignete Skalierung, k-Werte oder Clusterformen entstehen.")

print(f"\n🏭 BUSINESS-NUTZEN:")
profile_recommendations = {
    'Dauerbetrieb': 'Kapazitätsreserven und geplante Stillstandsfenster prüfen',
    'Teilzeit': 'Einsatzplanung und Möglichkeiten zur Lastverteilung prüfen',
    'Niedrige Nutzung': 'Ursachen prüfen, z. B. geringe Nachfrage oder geplanten Stillstand'
}
for i, profile in enumerate(labels_machines):
    print(f"• Cluster {i} ({profile}): {profile_recommendations[profile]}")
print("Wartungsbedarf erfordert zusätzliche Informationen, etwa Störmeldungen und Zustandsdaten.")

## Zusammenfassung und Interpretation

Der Silhouette Score bewertet die geometrische Trennung und Kompaktheit einer Cluster-Zuordnung unter dem gewählten Distanzmaß. Er ersetzt keine fachliche Validierung und bevorzugt oft kompakte, konvexe Gruppen.

Die folgende Übersicht illustriert unterschiedliche Beispiele, ist aber keine Rangliste von Datensätzen: Die Maschinenprofile werden im standardisierten Merkmalsraum bewertet, die künstlichen Punktwolken in ihren ursprünglichen Koordinaten. Besonders aussagekräftig sind Vergleiche verschiedener Zuordnungen auf denselben Daten mit gleicher Vorverarbeitung und gleichem Distanzmaß.

In [ ]:
# Vergleich aller Szenarien
scenarios = [
    ('Perfekt getrennte Cluster', sil_score_perfect),
    ('Überlappende Cluster', sil_score_overlap),
    ('Falsche Cluster-Anzahl', sil_score_wrong),
    ('Maschinennutzungsprofile', sil_score_machines)
]

print("📊 SILHOUETTE SCORE VERGLEICH")
print("=" * 50)

for scenario, score in scenarios:
    quality = silhouette_quality(score)
    print(f"{scenario:.<30} {score:.3f} {quality}")

print("\n🎯 PRAKTISCHE EMPFEHLUNGEN:")
print("• Ab 0.7: Sehr starke Trennung; fachliche Bedeutung trotzdem prüfen")
print("• 0.5 bis <0.7: Deutliche Trennung; einzelne Cluster und Punkte untersuchen")
print("• 0.25 bis <0.5: Schwache bis mäßige Trennung; Skalierung und k-Werte prüfen")
print("• Unter 0.25: Geringe oder ungünstige Trennung unter der gewählten Darstellung")
print("\n💡 Clusterformen, Stabilität und Fachwissen zusätzlich berücksichtigen; ein niedriger Score schließt Struktur nicht aus.")